# Kavach 01: fine-tune YOLO11s on SH17 (PPE detection)

**Settings:** Accelerator **GPU T4 x2** (one GPU is used), Internet **ON**.
**Input:** add the dataset `mugheesahmad/sh17-dataset-for-ppe-detection`.
**Run:** *Save Version → Save & Run All (Commit)*. Training is time-boxed to 1.25 h.
**Output:** `/kaggle/working/kavach_ppe_out.zip` (best.pt, metrics JSON, plots).

In [ ]:
!pip install -q -U ultralytics
import os, json, random, shutil, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import yaml
from PIL import Image
T0 = time.time()

## 1. Inspect the dataset layout (don't guess paths or class order)

In [ ]:
ROOT = Path("/kaggle/input")
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
imgs = [p for p in ROOT.rglob("*") if p.suffix.lower() in IMG_EXT]
lbls = [p for p in ROOT.rglob("*.txt") if "labels" in p.parts]
splits = {p.name: p for p in ROOT.rglob("*.txt") if p.name in ("train_files.txt", "val_files.txt")}
yamls = list(ROOT.rglob("*.yaml")) + list(ROOT.rglob("*.yml"))
print("images:", len(imgs), "| label files:", len(lbls), "| split files:", list(splits), "| yamls:", yamls)
print("example image:", imgs[0]); print("example label:", lbls[0])
for name, p in splits.items():
    print(name, "first lines:", p.read_text().splitlines()[:3])

In [ ]:
# Class order: take it from the dataset's own yaml if one exists, else the SH17 repo's sh17.yaml order.
SH17_REPO_ORDER = ["person", "ear", "ear-mufs", "face", "face-guard", "face-mask", "foot", "tool",
                   "glasses", "gloves", "helmet", "hands", "head", "medical-suit", "shoes", "safety-suit", "safety-vest"]
names = None
for y in yamls:
    try:
        d = yaml.safe_load(open(y))
        n = d.get("names") if isinstance(d, dict) else None
        if n:
            names = list(n.values()) if isinstance(n, dict) else list(n)
            print("class names from", y); break
    except Exception as e:
        print("could not read", y, e)
if names is None:
    print("No yaml in the dataset, so using the SH17 repo order. CHECK the visualisation below.")
    names = SH17_REPO_ORDER
for i, n in enumerate(names): print(i, n)
assert len(names) == 17, len(names)

## 2. Build train/val lists and pre-resize to a 960 px long side (originals reach 8192 px)

In [ ]:
by_stem = {p.stem: p for p in imgs}
lbl_by_stem = {p.stem: p for p in lbls}
def stems_from(split_file):
    return [Path(l.strip()).stem for l in split_file.read_text().splitlines() if l.strip()]
train_stems = [s for s in stems_from(splits["train_files.txt"]) if s in by_stem and s in lbl_by_stem]
val_stems = [s for s in stems_from(splits["val_files.txt"]) if s in by_stem and s in lbl_by_stem]
print("train:", len(train_stems), "val:", len(val_stems))
assert len(train_stems) > 1000 and len(val_stems) > 200

OUT = Path("/kaggle/working/sh17")
MAX_SIDE = 960
def convert(args):
    stem, split = args
    try:
        im = Image.open(by_stem[stem]); im.draft("RGB", (MAX_SIDE, MAX_SIDE))  # fast JPEG downscale while decoding
        im = im.convert("RGB"); im.thumbnail((MAX_SIDE, MAX_SIDE))
        im.save(OUT / "images" / split / f"{stem}.jpg", quality=92)
        shutil.copy(lbl_by_stem[stem], OUT / "labels" / split / f"{stem}.txt")  # YOLO labels are normalised
        return True
    except Exception as e:
        print("skip", stem, e); return False
for split in ("train", "val"):
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True); (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)
jobs = [(s, "train") for s in train_stems] + [(s, "val") for s in val_stems]
with ProcessPoolExecutor(os.cpu_count()) as ex:
    ok = sum(ex.map(convert, jobs, chunksize=32))
print(f"converted {ok}/{len(jobs)} in {time.time()-T0:.0f}s")

In [ ]:
# Sanity check: draw labels on 6 train images. Helmets must say "helmet", vests "safety-vest".
import cv2, matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, stem in zip(axes.flat, random.Random(0).sample(train_stems, 6)):
    im = cv2.imread(str(OUT / "images/train" / f"{stem}.jpg"))[:, :, ::-1].copy(); h, w = im.shape[:2]
    for line in open(OUT / "labels/train" / f"{stem}.txt"):
        c, x, y, bw, bh = map(float, line.split()[:5])
        x1, y1, x2, y2 = int((x - bw/2) * w), int((y - bh/2) * h), int((x + bw/2) * w), int((y + bh/2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(im, names[int(c)], (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    ax.imshow(im); ax.axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/label_check.jpg", dpi=60); plt.show()

## 3. Train (time-boxed: Ultralytics fits the LR schedule to the time budget)

In [ ]:
data = {"path": str(OUT), "train": "images/train", "val": "images/val", "names": {i: n for i, n in enumerate(names)}}
yaml.safe_dump(data, open("/kaggle/working/sh17.yaml", "w"), sort_keys=False)
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(data="/kaggle/working/sh17.yaml", imgsz=640, epochs=100, time=1.25, batch=32, device=0, workers=4,
            cache="ram", cos_lr=True, close_mosaic=5, patience=100, seed=0, plots=True,
            project="/kaggle/working/runs", name="sh17_y11s", exist_ok=True)

## 4. Validate and package the outputs

In [ ]:
run = Path("/kaggle/working/runs/sh17_y11s")
best = YOLO(str(run / "weights/best.pt"))
m = best.val(data="/kaggle/working/sh17.yaml", imgsz=640, batch=32, device=0, split="val", plots=True,
             project="/kaggle/working/runs", name="val", exist_ok=True)
per_class = {names[int(c)]: {"ap50": float(m.box.ap50[i]), "ap50_95": float(m.box.ap[i])}
             for i, c in enumerate(m.box.ap_class_index)}
metrics = {"model": "yolo11s", "dataset": "SH17 (official train/val split)", "imgsz": 640,
           "train_hours_budget": 1.25, "epochs_completed": len(open(run / "results.csv").read().splitlines()) - 1,
           "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map), "precision": float(m.box.mp),
           "recall": float(m.box.mr), "per_class": per_class}
print(json.dumps(metrics, indent=2))
out = Path("/kaggle/working/out"); out.mkdir(exist_ok=True)
json.dump(metrics, open(out / "ppe_val_metrics.json", "w"), indent=2)
shutil.copy(run / "weights/best.pt", out / "ppe_y11s_sh17.pt"); shutil.copy(run / "results.csv", out)
for p in list(run.glob("*.png")) + sorted(run.glob("val_batch*_pred.jpg"))[:3] + list(Path("/kaggle/working/runs/val").glob("*.png")):
    shutil.copy(p, out / p.name)
shutil.make_archive("/kaggle/working/kavach_ppe_out", "zip", out)
shutil.rmtree(OUT)  # don't keep the resized images in the saved output
print(f"done in {(time.time() - T0) / 60:.0f} min -> /kaggle/working/kavach_ppe_out.zip")